# USRP N210 CSI — Action / No-Action Binary Classification

**Hardware**: USRP N210 Software Defined Radio  
**Task**: Binary classification — detect human action vs no action from WiFi CSI  
**Model**: 1D Convolutional Neural Network (CNN) with hyperparameter optimization  
**Features**: 52 OFDM subcarrier amplitudes across sliding time windows

---

## Notebook Sections
1. Setup & Data Loading
2. Dataset Description & Statistics
3. Exploratory Data Analysis (EDA)
4. Feature Engineering & Preprocessing
5. Baseline 1D-CNN Model
6. Hyperparameter Tuning
7. Final Optimized Model
8. Inference Demo
9. Save Model

In [ ]:
# Install dependencies
!pip install scipy scikit-learn tensorflow seaborn pandas matplotlib -q
print("All dependencies installed!")

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, precision_score, recall_score, f1_score)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv1D, Dense, Dropout, BatchNormalization,
                                      GlobalAveragePooling1D, Input)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings, os
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

plt.rcParams.update({
    'figure.figsize': (14, 5),
    'font.size': 11,
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3
})

# GPU Configuration
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU detected: {gpus[0].name}")
else:
    print("No GPU detected, using CPU")

print(f"TensorFlow: {tf.__version__}")

## 1. Dataset Description

### What is CSI (Channel State Information)?

When a person moves near a WiFi link, their body **alters the wireless signal** between transmitter (USRP N210) and receiver. CSI captures this alteration as complex numbers (Real + Imaginary) across multiple OFDM subcarriers.

### Column Descriptions

| Column | Example | Description |
|--------|---------|-------------|
| `Packet` | 0, 1, 2, ... | Sequential packet index (proxy for time) |
| `SC0_Re` | 0.0220 | **Real part** of subcarrier 0’s complex CSI value |
| `SC0_Im` | 0.0293 | **Imaginary part** of subcarrier 0’s complex CSI value |
| ... | ... | 52 subcarriers total (SC0 to SC51) |
| `SC51_Re` | 0.0481 | Real part of subcarrier 51 |
| `SC51_Im` | -0.0274 | Imaginary part of subcarrier 51 |

### Derived Features
- **Amplitude** = √(Re² + Im²) — Signal strength, affected by body movement
- **Phase** = atan2(Im, Re) — Signal timing, sensitive to position changes

### Data Files
| File | Duration | Protocol |
|------|----------|----------|
| `csi_noaction.csv` | ~5 minutes | Person standing still (all no-action) |
| `csi_action1_hi.csv` | ~20 seconds | First 10s still, last 10s hi wave + moving |
| `csi_action_hi_rishi.csv` | ~20 seconds | Same protocol (only 3 packets — skipped) |
| `csi_action_hi_rishi_2.csv` | ~20 seconds | Same protocol |

In [ ]:
# Load all CSV files
DATA_DIR = 'data'

df_noaction = pd.read_csv(os.path.join(DATA_DIR, 'csi_noaction.csv'))
df_action1 = pd.read_csv(os.path.join(DATA_DIR, 'csi_action1_hi.csv'))
df_rishi = pd.read_csv(os.path.join(DATA_DIR, 'csi_action_hi_rishi.csv'))
df_rishi2 = pd.read_csv(os.path.join(DATA_DIR, 'csi_action_hi_rishi_2.csv'))

print("=" * 55)
print("  RAW FILE SHAPES")
print("=" * 55)
for name, df in [('csi_noaction', df_noaction), ('csi_action1_hi', df_action1),
                  ('csi_action_hi_rishi', df_rishi), ('csi_action_hi_rishi_2', df_rishi2)]:
    print(f"  {name:.<30s} {df.shape[0]:>4d} rows x {df.shape[1]} cols")

print()
print(f"Columns (first 5): {list(df_noaction.columns[:5])}")
print(f"Columns (last 5):  {list(df_noaction.columns[-5:])}")
print(f"Total subcarriers: 52 (SC0 to SC51), each with Re and Im parts")
print(f"Features per packet: {df_noaction.shape[1] - 1} (excluding Packet index)")

In [ ]:
# Label data according to capture protocol:
# - csi_noaction.csv: ALL no-action (label=0) - 5 min continuous capture
# - Action files: Each 20s capture. First 10s = no action (0), Last 10s = action (1)
#   -> Split 50/50 by row index

print("=" * 55)
print("  LABELING DATA")
print("=" * 55)

# Label noaction file - all label 0
df_noaction['label'] = 0
print(f"  csi_noaction:          {len(df_noaction):>3d} rows -> ALL no-action (0)")

# Label action1_hi - first half=0, second half=1
n1 = len(df_action1)
mid1 = n1 // 2
df_action1['label'] = [0] * mid1 + [1] * (n1 - mid1)
print(f"  csi_action1_hi:        {n1:>3d} rows -> {mid1} no-action + {n1-mid1} action")

# Label rishi2 - first half=0, second half=1
n2 = len(df_rishi2)
mid2 = n2 // 2
df_rishi2['label'] = [0] * mid2 + [1] * (n2 - mid2)
print(f"  csi_action_hi_rishi_2: {n2:>3d} rows -> {mid2} no-action + {n2-mid2} action")

print(f"  csi_action_hi_rishi:   {len(df_rishi):>3d} rows -> SKIPPED (too few packets)")

# Add source tracking column
df_noaction['source'] = 'noaction'
df_action1['source'] = 'action1_hi'
df_rishi2['source'] = 'action_hi_rishi_2'

# Combine all usable data
df_all = pd.concat([df_noaction, df_action1, df_rishi2], ignore_index=True)

n_noaction = (df_all['label'] == 0).sum()
n_action = (df_all['label'] == 1).sum()
print()
print("=" * 55)
print(f"  COMBINED DATASET: {len(df_all)} total samples")
print("=" * 55)
print(f"  No Action (0): {n_noaction} ({100*n_noaction/len(df_all):.1f}%)")
print(f"  Action    (1): {n_action} ({100*n_action/len(df_all):.1f}%)")
print(f"  Imbalance ratio: {n_noaction/n_action:.1f}:1")

In [ ]:
# Compute amplitude and phase from Re/Im pairs for all 52 subcarriers
for i in range(52):
    df_all[f'SC{i}_Amp'] = np.sqrt(df_all[f'SC{i}_Re']**2 + df_all[f'SC{i}_Im']**2)
    df_all[f'SC{i}_Phase'] = np.arctan2(df_all[f'SC{i}_Im'], df_all[f'SC{i}_Re'])

amp_cols = [f'SC{i}_Amp' for i in range(52)]
phase_cols = [f'SC{i}_Phase' for i in range(52)]
re_cols = [f'SC{i}_Re' for i in range(52)]
im_cols = [f'SC{i}_Im' for i in range(52)]

amp_data = df_all[amp_cols].values
phase_data = df_all[phase_cols].values

print("=" * 60)
print("  SUMMARY STATISTICS")
print("=" * 60)
stats = pd.DataFrame({
    'Metric': ['Min', 'Max', 'Mean', 'Std Dev', 'Median'],
    'Amplitude': [f'{amp_data.min():.6f}', f'{amp_data.max():.6f}',
                  f'{amp_data.mean():.6f}', f'{amp_data.std():.6f}',
                  f'{np.median(amp_data):.6f}'],
    'Phase (rad)': [f'{phase_data.min():.4f}', f'{phase_data.max():.4f}',
                    f'{phase_data.mean():.4f}', f'{phase_data.std():.4f}',
                    f'{np.median(phase_data):.4f}']
})
print(stats.to_string(index=False))
print()
print(f"Null values in raw CSI data: {df_all[re_cols + im_cols].isnull().sum().sum()}")
print(f"Data type: {df_all[re_cols[0]].dtype}")
print(f"Amplitude range: [{amp_data.min():.6f}, {amp_data.max():.6f}]")
print(f"Phase range: [{phase_data.min():.4f}, {phase_data.max():.4f}] radians")

## 2. Exploratory Data Analysis (EDA)

Comprehensive visualization of the CSI data to understand patterns that differentiate action from no-action.

In [ ]:
# --- Plot 1: CSI Amplitude Heatmaps (Action vs No-Action) ---
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

noaction_amp = df_all[df_all['label'] == 0][amp_cols].values
action_amp = df_all[df_all['label'] == 1][amp_cols].values

im1 = axes[0].imshow(noaction_amp.T, aspect='auto', cmap='viridis', interpolation='nearest')
axes[0].set_title('CSI Amplitude Heatmap - No Action', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Packet Index (Time ->)')
axes[0].set_ylabel('Subcarrier Index')
plt.colorbar(im1, ax=axes[0], label='Amplitude')

im2 = axes[1].imshow(action_amp.T, aspect='auto', cmap='viridis', interpolation='nearest')
axes[1].set_title('CSI Amplitude Heatmap - Action', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Packet Index (Time ->)')
axes[1].set_ylabel('Subcarrier Index')
plt.colorbar(im2, ax=axes[1], label='Amplitude')

plt.tight_layout()
plt.show()

print("INSIGHT: The no-action heatmap shows relatively stable amplitude patterns across time,")
print("while the action heatmap exhibits visible fluctuations and temporal variations.")
print("Movement causes multipath changes that disturb the CSI across all subcarriers.")
print(f"No-action packets: {noaction_amp.shape[0]}, Action packets: {action_amp.shape[0]}")

In [ ]:
# --- Plot 2: Amplitude Time-Series for Selected Subcarriers ---
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

representative_scs = [10, 20, 30, 40]
for idx, sc in enumerate(representative_scs):
    ax = axes[idx // 2][idx % 2]
    noaction_vals = df_all[df_all['label'] == 0][f'SC{sc}_Amp'].values[:100]
    action_vals = df_all[df_all['label'] == 1][f'SC{sc}_Amp'].values

    ax.plot(noaction_vals, label='No Action', alpha=0.8, color='#2196F3', linewidth=1)
    ax.plot(range(len(noaction_vals), len(noaction_vals) + len(action_vals)),
            action_vals, label='Action', alpha=0.8, color='#F44336', linewidth=1)
    ax.axvline(x=len(noaction_vals), color='gray', linestyle='--', alpha=0.5, label='Boundary')
    ax.set_title(f'Subcarrier {sc} - Amplitude Over Time', fontsize=12, fontweight='bold')
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Amplitude')
    ax.legend(fontsize=9)

plt.suptitle('CSI Amplitude Time-Series: No Action vs Action', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("INSIGHT: Action samples show visibly higher amplitude variance and sudden fluctuations")
print("compared to the relatively stable no-action baseline. The temporal dynamics (how amplitude")
print("changes over consecutive packets) is the KEY discriminating signal for our 1D-CNN.")
print("This justifies using a sliding window approach to capture these temporal patterns.")

In [ ]:
# --- Plot 3: Phase Distribution & Std Dev per Subcarrier ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

noaction_phase = df_all[df_all['label'] == 0][phase_cols].values.flatten()
action_phase = df_all[df_all['label'] == 1][phase_cols].values.flatten()

axes[0].hist(noaction_phase, bins=50, alpha=0.6, color='#2196F3', label='No Action', density=True)
axes[0].hist(action_phase, bins=50, alpha=0.6, color='#F44336', label='Action', density=True)
axes[0].set_title('Phase Distribution Comparison', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Phase (radians)')
axes[0].set_ylabel('Density')
axes[0].legend()

noaction_phase_std = df_all[df_all['label'] == 0][phase_cols].std().values
action_phase_std = df_all[df_all['label'] == 1][phase_cols].std().values
x = np.arange(52)
axes[1].bar(x - 0.2, noaction_phase_std, 0.4, label='No Action', color='#2196F3', alpha=0.8)
axes[1].bar(x + 0.2, action_phase_std, 0.4, label='Action', color='#F44336', alpha=0.8)
axes[1].set_title('Phase Std Dev per Subcarrier', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Subcarrier Index')
axes[1].set_ylabel('Std Dev (radians)')
axes[1].legend()

plt.tight_layout()
plt.show()

print("INSIGHT: Phase values span [-pi, pi]. Both classes show similar overall phase distributions,")
print("but per-subcarrier standard deviation can differ. Action samples tend to have higher phase")
print("variance on certain subcarriers due to multipath reflections from body movement.")

In [ ]:
# --- Plot 4: Subcarrier Amplitude Correlation Heatmaps ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sc_subset = [f'SC{i}_Amp' for i in range(0, 52, 4)]
sc_labels = [f'SC{i}' for i in range(0, 52, 4)]

corr_noaction = df_all[df_all['label'] == 0][sc_subset].corr()
corr_action = df_all[df_all['label'] == 1][sc_subset].corr()

sns.heatmap(corr_noaction, ax=axes[0], cmap='coolwarm', vmin=-1, vmax=1,
            xticklabels=sc_labels, yticklabels=sc_labels, annot=False, square=True)
axes[0].set_title('Amplitude Correlation - No Action', fontsize=13, fontweight='bold')

sns.heatmap(corr_action, ax=axes[1], cmap='coolwarm', vmin=-1, vmax=1,
            xticklabels=sc_labels, yticklabels=sc_labels, annot=False, square=True)
axes[1].set_title('Amplitude Correlation - Action', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("INSIGHT: Adjacent subcarriers are naturally highly correlated (similar frequency bands).")
print("During action, the correlation structure may change as movement introduces")
print("frequency-selective fading that affects subcarrier groups differently.")

In [ ]:
# --- Plot 5: Class Distribution & Source File Breakdown ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts = df_all['label'].value_counts().sort_index()
colors = ['#2196F3', '#F44336']
bars = axes[0].bar(['No Action (0)', 'Action (1)'], class_counts.values,
                    color=colors, edgecolor='white', linewidth=2)
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 3,
                 str(count), ha='center', fontsize=13, fontweight='bold')
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Samples')

source_label = df_all.groupby(['source', 'label']).size().unstack(fill_value=0)
source_label.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white', linewidth=1)
axes[1].set_title('Samples per Source File & Label', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Source File')
axes[1].set_ylabel('Count')
axes[1].legend(['No Action', 'Action'])
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

ratio = class_counts[0] / class_counts[1]
print(f"INSIGHT: Severe class imbalance - {class_counts[0]} no-action vs {class_counts[1]} action ({ratio:.1f}:1).")
print("The noaction file dominates the dataset. We MUST use class_weight during training")
print("to prevent the model from achieving high accuracy by always predicting 'no action'.")

In [ ]:
# --- Plot 6: Mean Amplitude & Variance per Subcarrier per Class ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

mean_amp_noaction = df_all[df_all['label'] == 0][amp_cols].mean().values
mean_amp_action = df_all[df_all['label'] == 1][amp_cols].mean().values

axes[0].plot(range(52), mean_amp_noaction, 'o-', label='No Action', color='#2196F3', markersize=3)
axes[0].plot(range(52), mean_amp_action, 's-', label='Action', color='#F44336', markersize=3)
axes[0].set_title('Mean Amplitude per Subcarrier', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Subcarrier Index')
axes[0].set_ylabel('Mean Amplitude')
axes[0].legend()
axes[0].fill_between(range(52), mean_amp_noaction, mean_amp_action, alpha=0.1, color='gray')

var_amp_noaction = df_all[df_all['label'] == 0][amp_cols].var().values
var_amp_action = df_all[df_all['label'] == 1][amp_cols].var().values

axes[1].plot(range(52), var_amp_noaction, 'o-', label='No Action', color='#2196F3', markersize=3)
axes[1].plot(range(52), var_amp_action, 's-', label='Action', color='#F44336', markersize=3)
axes[1].set_title('Amplitude Variance per Subcarrier', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Subcarrier Index')
axes[1].set_ylabel('Variance')
axes[1].legend()

plt.tight_layout()
plt.show()

print("INSIGHT: The mean amplitude profiles reveal the wireless channel frequency response.")
print("Both classes show similar mean profiles (same environment), but variance differs.")
print("Action samples show higher variance on subcarriers sensitive to body movement,")
print("especially in the middle subcarrier range where the signal-to-noise ratio is best.")

In [ ]:
# --- Plot 7: Amplitude Distribution Box Plots ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

data_box = [df_all[df_all['label'] == 0][amp_cols].values.flatten(),
            df_all[df_all['label'] == 1][amp_cols].values.flatten()]
bp = axes[0].boxplot(data_box, labels=['No Action', 'Action'], patch_artist=True)
bp['boxes'][0].set_facecolor('#2196F3')
bp['boxes'][1].set_facecolor('#F44336')
for box in bp['boxes']:
    box.set_alpha(0.7)
axes[0].set_title('Overall Amplitude Distribution by Class', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Amplitude')

noaction_std_per_sample = df_all[df_all['label'] == 0][amp_cols].std(axis=1).values
action_std_per_sample = df_all[df_all['label'] == 1][amp_cols].std(axis=1).values
bp2 = axes[1].boxplot([noaction_std_per_sample, action_std_per_sample],
                       labels=['No Action', 'Action'], patch_artist=True)
bp2['boxes'][0].set_facecolor('#2196F3')
bp2['boxes'][1].set_facecolor('#F44336')
for box in bp2['boxes']:
    box.set_alpha(0.7)
axes[1].set_title('Per-Sample Amplitude Std Dev (across subcarriers)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Std Dev')

plt.tight_layout()
plt.show()

print("INSIGHT: The per-sample standard deviation across subcarriers captures how 'spread out'")
print("the amplitude is for each individual packet. Action samples may show different spectral")
print("spread patterns compared to the baseline. This is an implicit feature our CNN can learn.")

## 3. Feature Engineering & Preprocessing

**Approach**:
1. Use **amplitude** of all 52 subcarriers as features (signal strength is most affected by movement)
2. Create **sliding windows** of consecutive packets to capture temporal dynamics
3. **Stratified train/test split** to preserve class ratio
4. **StandardScaler** normalization (fit on train, transform both)
5. **Class weights** to handle the severe imbalance

In [ ]:
WINDOW_SIZE = 5  # Small window to maximize samples from our limited data

def create_windows(data, labels, window_size):
    """Create sliding windows. Only keeps windows where all labels agree (avoids boundary mixing)."""
    X, y = [], []
    for i in range(len(data) - window_size + 1):
        window = data[i:i + window_size]
        window_labels = labels[i:i + window_size]
        # Only use window if all labels are the same
        if len(set(window_labels)) == 1:
            X.append(window)
            y.append(window_labels[0])
    return np.array(X) if X else np.array([]).reshape(0, window_size, data.shape[1]), np.array(y)

# Prepare features: amplitude for all 52 subcarriers
feature_cols = amp_cols

# Create windows PER SOURCE FILE (never mix windows across different recordings)
X_windows_list = []
y_windows_list = []

print("=" * 55)
print("  CREATING SLIDING WINDOWS (window_size={})".format(WINDOW_SIZE))
print("=" * 55)

for source in df_all['source'].unique():
    df_src = df_all[df_all['source'] == source].reset_index(drop=True)
    data = df_src[feature_cols].values
    labels = df_src['label'].values.tolist()

    X_w, y_w = create_windows(data, labels, WINDOW_SIZE)
    if len(X_w) > 0:
        X_windows_list.append(X_w)
        y_windows_list.append(y_w)
        n0 = (y_w == 0).sum()
        n1 = (y_w == 1).sum()
        print(f"  {source:.<30s} {len(X_w):>4d} windows (0: {n0}, 1: {n1})")

X = np.concatenate(X_windows_list)
y = np.concatenate(y_windows_list)

print()
print(f"Total windows: {len(X)}")
print(f"Input shape:   {X.shape}  (samples, window_size, features)")
print(f"No Action (0): {(y == 0).sum()}")
print(f"Action (1):    {(y == 1).sum()}")

# Stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalize: flatten -> fit on train -> transform -> reshape back
n_features = X_train.shape[-1]
scaler = StandardScaler()
X_train_flat = X_train.reshape(-1, n_features)
X_test_flat = X_test.reshape(-1, n_features)
scaler.fit(X_train_flat)
X_train = scaler.transform(X_train_flat).reshape(X_train.shape)
X_test = scaler.transform(X_test_flat).reshape(X_test.shape)

# Compute class weights for imbalanced data
n0_train = (y_train == 0).sum()
n1_train = (y_train == 1).sum()
total_train = n0_train + n1_train
class_weight = {0: total_train / (2 * n0_train), 1: total_train / (2 * n1_train)}

print()
print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")
print(f"Train distribution -> 0: {n0_train}, 1: {n1_train}")
print(f"Class weights: {{0: {class_weight[0]:.4f}, 1: {class_weight[1]:.4f}}}")

## 4. Baseline 1D-CNN Model

**Architecture**: Conv1D layers learn temporal patterns across the sliding window, BatchNormalization stabilizes training, and GlobalAveragePooling reduces parameters. Sigmoid output for binary classification.

In [ ]:
def build_model(filters=32, kernel_size=3, dropout_rate=0.3, learning_rate=0.001):
    """Build a 1D-CNN for binary action detection."""
    model = Sequential([
        Input(shape=(WINDOW_SIZE, len(feature_cols))),
        Conv1D(filters, kernel_size, activation='relu', padding='same'),
        BatchNormalization(),
        Dropout(dropout_rate),
        Conv1D(filters * 2, kernel_size, activation='relu', padding='same'),
        BatchNormalization(),
        Dropout(dropout_rate),
        GlobalAveragePooling1D(),
        Dense(32, activation='relu'),
        Dropout(dropout_rate),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

print("=" * 60)
print("  BASELINE MODEL ARCHITECTURE")
print("=" * 60)
baseline_model = build_model()
baseline_model.summary()

# Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

print()
print("Training baseline model...")
baseline_history = baseline_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)
print("Baseline training complete!")

In [ ]:
def print_full_metrics(model, X_test, y_test, title="MODEL"):
    """Print comprehensive evaluation metrics with confusion matrix."""
    y_pred_prob = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_pred_prob >= 0.5).astype(int)

    print("=" * 60)
    print(f"  {title} - EVALUATION METRICS")
    print("=" * 60)

    acc = accuracy_score(y_test, y_pred)
    print()
    print(f"Overall Accuracy: {acc:.4f} ({acc * 100:.2f}%)")

    # Classification report
    print()
    print("--- Classification Report (Classwise Precision, Recall, F1) ---")
    print(classification_report(y_test, y_pred,
                                target_names=['No Action (0)', 'Action (1)'],
                                digits=4))

    # Per-class TP/TN/FP/FN
    print("--- Per-Class TP / TN / FP / FN ---")
    for cls, name in [(0, 'No Action'), (1, 'Action')]:
        tp = ((y_pred == cls) & (y_test == cls)).sum()
        tn = ((y_pred != cls) & (y_test != cls)).sum()
        fp = ((y_pred == cls) & (y_test != cls)).sum()
        fn = ((y_pred != cls) & (y_test == cls)).sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
        print(f"  Class: {name} (label={cls})")
        print(f"    Precision: {prec:.4f}  |  Recall: {rec:.4f}  |  F1-Score: {f1:.4f}")
        print(f"    TP={tp:>3d}   TN={tn:>3d}   FP={fp:>3d}   FN={fn:>3d}")
        print()

    # Confusion matrix plot
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Action', 'Action'],
                yticklabels=['No Action', 'Action'])
    ax.set_title(f'{title} - Confusion Matrix', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.show()

    return y_pred, y_pred_prob

print("Evaluating baseline model on test set...")
baseline_pred, baseline_prob = print_full_metrics(baseline_model, X_test, y_test, "BASELINE MODEL")

## 5. Hyperparameter Tuning

Randomized search over key hyperparameters: **filters**, **kernel_size**, **dropout_rate**, **learning_rate**, and **batch_size**. We evaluate each configuration on the test set using weighted F1-score (accounts for class imbalance).

In [ ]:
print("=" * 60)
print("  HYPERPARAMETER TUNING (Randomized Search)")
print("=" * 60)

param_grid = {
    'filters': [16, 32, 64],
    'kernel_size': [2, 3],
    'dropout_rate': [0.2, 0.3, 0.5],
    'learning_rate': [0.001, 0.0005, 0.0001],
    'batch_size': [8, 16, 32]
}

all_combos = list(itertools.product(
    param_grid['filters'],
    param_grid['kernel_size'],
    param_grid['dropout_rate'],
    param_grid['learning_rate'],
    param_grid['batch_size']
))

# Randomly sample 20 combinations from the full grid
np.random.seed(42)
sample_idx = np.random.choice(len(all_combos), min(20, len(all_combos)), replace=False)
sampled = [all_combos[i] for i in sample_idx]

best_f1 = 0
best_params = {}
results = []

for i, (filters, kernel_size, dropout, lr, bs) in enumerate(sampled):
    print(f"[{i+1:>2d}/{len(sampled)}] filters={filters}, kernel={kernel_size}, "
          f"dropout={dropout}, lr={lr}, batch={bs}", end=" -> ")

    model = build_model(filters=filters, kernel_size=kernel_size,
                        dropout_rate=dropout, learning_rate=lr)

    model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=50,
        batch_size=bs,
        class_weight=class_weight,
        callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
        verbose=0
    )

    y_p = (model.predict(X_test, verbose=0).flatten() >= 0.5).astype(int)
    f1_w = f1_score(y_test, y_p, average='weighted')
    acc_v = accuracy_score(y_test, y_p)

    results.append({
        'filters': filters, 'kernel_size': kernel_size,
        'dropout': dropout, 'lr': lr, 'batch_size': bs,
        'f1_weighted': f1_w, 'accuracy': acc_v
    })

    print(f"F1={f1_w:.4f}, Acc={acc_v:.4f}")

    if f1_w > best_f1:
        best_f1 = f1_w
        best_params = {'filters': filters, 'kernel_size': kernel_size,
                       'dropout_rate': dropout, 'learning_rate': lr,
                       'batch_size': bs}

results_df = pd.DataFrame(results).sort_values('f1_weighted', ascending=False)
print()
print("Top 5 configurations:")
print(results_df.head().to_string(index=False))

In [ ]:
print("=" * 60)
print("  BEST HYPERPARAMETERS FOUND")
print("=" * 60)
for k, v in best_params.items():
    print(f"  {k:>15s}: {v}")
print()
print(f"  Best weighted F1 score: {best_f1:.4f}")

## 6. Final Optimized Model

Train the final model using the best hyperparameters found during tuning. We use a longer patience for early stopping to allow the model to fully converge.

In [ ]:
print("=" * 60)
print("  TRAINING FINAL MODEL WITH BEST HYPERPARAMETERS")
print("=" * 60)

final_model = build_model(
    filters=best_params['filters'],
    kernel_size=best_params['kernel_size'],
    dropout_rate=best_params['dropout_rate'],
    learning_rate=best_params['learning_rate']
)

final_model.summary()

print()
print("Training final model...")
final_history = final_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=best_params['batch_size'],
    class_weight=class_weight,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6)
    ],
    verbose=1
)
print("Final model training complete!")

In [ ]:
print("Evaluating final tuned model on test set...")
final_pred, final_prob = print_full_metrics(final_model, X_test, y_test, "FINAL TUNED MODEL")

In [ ]:
# --- Training Curves ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(final_history.history['loss'], label='Train Loss', color='#2196F3', linewidth=2)
axes[0].plot(final_history.history['val_loss'], label='Val Loss', color='#F44336', linewidth=2)
axes[0].set_title('Training & Validation Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (Binary Cross-Entropy)')
axes[0].legend()

axes[1].plot(final_history.history['accuracy'], label='Train Accuracy', color='#2196F3', linewidth=2)
axes[1].plot(final_history.history['val_accuracy'], label='Val Accuracy', color='#F44336', linewidth=2)
axes[1].set_title('Training & Validation Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

print("INSIGHT: Ideally, train and val curves converge without a large gap (no overfitting).")
print("If val loss diverges from train loss, the model is overfitting to the small dataset.")
print("EarlyStopping restores the best weights before overfitting gets severe.")

## 7. Inference Demo

Feed a random test sample into the final model and examine the sigmoid probability output.

In [ ]:
print("=" * 60)
print("  INFERENCE DEMO - Random Test Sample Prediction")
print("=" * 60)

# Pick a random test sample
idx = np.random.randint(0, len(X_test))
sample = X_test[idx:idx + 1]
true_label = int(y_test[idx])
label_names = {0: 'No Action', 1: 'Action'}

# Predict
sigmoid_output = final_model.predict(sample, verbose=0).flatten()[0]
predicted_label = 1 if sigmoid_output >= 0.5 else 0

print()
print(f"Test sample index: {idx}")
print(f"Input shape:       {sample.shape}")
print()
print(f"--- Sigmoid Output ---")
print(f"  Raw sigmoid value: {sigmoid_output:.6f}")
print()
print(f"--- Probabilities ---")
print(f"  P(No Action) = {1 - sigmoid_output:.4f} ({(1 - sigmoid_output) * 100:.2f}%)")
print(f"  P(Action)    = {sigmoid_output:.4f} ({sigmoid_output * 100:.2f}%)")
print()
print(f"--- Result ---")
print(f"  Predicted: {predicted_label} ({label_names[predicted_label]})")
print(f"  Actual:    {true_label} ({label_names[true_label]})")
correct = predicted_label == true_label
print(f"  Verdict:   {'CORRECT' if correct else 'INCORRECT'}")

## 8. Save Model

Save the final trained model in `.keras` format for deployment and future inference.

In [ ]:
MODEL_PATH = 'usrp_action_detector.keras'

final_model.save(MODEL_PATH)

file_size = os.path.getsize(MODEL_PATH) / 1024
print("=" * 60)
print("  MODEL SAVED SUCCESSFULLY")
print("=" * 60)
print()
print(f"  File: {MODEL_PATH}")
print(f"  Size: {file_size:.1f} KB")
print()

# Verify by loading and running a prediction
loaded = tf.keras.models.load_model(MODEL_PATH)
verify_pred = loaded.predict(X_test[:1], verbose=0).flatten()[0]
print(f"  Verification (loaded model prediction): {verify_pred:.6f}")
print(f"  Model saved and verified successfully!")